# 02 — Split Generator (Run only after Gate 1 PASS)
Generates one stratified 70/15/15 split.csv with seed 42. Do not run before Dataset Audit approval.


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
AUDIT_OUT = Path('/content/drive/MyDrive/Deep Learning - RealWaste/04_Results/dataset_audit')
SPLIT_DIR = Path('/content/drive/MyDrive/Deep Learning - RealWaste/02_Splits')
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(AUDIT_OUT/'image_metadata.csv')
# Assumes any corrupted images were excluded from image_metadata.csv.


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df['class_name']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df['class_name']
)

train_df = train_df.copy(); train_df['split'] = 'train'
val_df = val_df.copy(); val_df['split'] = 'val'
test_df = test_df.copy(); test_df['split'] = 'test'

split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

class_names = sorted(split_df['class_name'].unique())
class_to_id = {name: i for i, name in enumerate(class_names)}
split_df['class_id'] = split_df['class_name'].map(class_to_id)

split_df[['filepath','class_name','class_id','split']].to_csv(SPLIT_DIR/'split.csv', index=False)

print(split_df['split'].value_counts())
print(pd.crosstab(split_df['class_name'], split_df['split']))
print('Saved:', SPLIT_DIR/'split.csv')


After generating the split:
1. Save the class mapping.
2. Verify every class exists in all three splits.
3. Never regenerate this split independently for another model.
4. Commit/copy the split file as a fixed project artifact.
